# STA 2503 · Project 1 — Notebook 00: Python warm-up

**Goal:** learn exactly the Python you need for Project 1 — nothing more. Every example comes from *your own* lecture/tutorial notes, so you already know the right answers.

### How this notebook works
- Run a cell with **Shift + Enter**. Run them top to bottom (later cells use variables from earlier ones).
- **Demo** cells are complete — just run them and read the output.
- **Exercise** cells contain blanks `____`. Replace each `____` with code, then run.
  - If you see `NameError: name '____' is not defined`, you missed a blank.
- Right after each exercise there's a **check** cell. It prints ✅ if you're right, or ❌ with a hint.
- Stuck for more than ~5 minutes? Ask me for a hint (not the answer) — tell me the exercise number.

### R → Python cheat sheet (keep this open)

| What | R | Python |
|---|---|---|
| assign | `x <- 3` | `x = 3` |
| power | `2^3` | `2**3`  ⚠️ `2^3` in Python is something else! |
| vector | `c(1, 2, 3)` | `np.array([1, 2, 3])` |
| first element | `x[1]` | `x[0]`  (Python counts from **0**) |
| last element | `x[length(x)]` | `x[-1]` |
| integer sequence | `1:5` | `np.arange(1, 6)`  (stop is **excluded**) |
| grid | `seq(0, 1, length.out = 11)` | `np.linspace(0, 1, 11)` |
| cumulative sum | `cumsum(x)` | `np.cumsum(x)` |
| elementwise max | `pmax(x, 0)` | `np.maximum(x, 0)` |
| random seed + normals | `set.seed(1); rnorm(5)` | `rng = np.random.default_rng(1); rng.normal(size=5)` |
| coin flips | `sample(c(-1,1), 10, replace=TRUE)` | `rng.choice([-1, 1], size=10)` |
| function | `f <- function(x) { x^2 }` | `def f(x):`<br>`    return x**2` |
| loop | `for (i in 1:n) {...}` | `for i in range(n):` → i = 0, 1, …, n-1 |
| row / column ops | `apply(M, 1, sum)` / `apply(M, 2, sum)` | `M.sum(axis=1)` / `M.sum(axis=0)` |

In [ ]:
# DEMO — run this first. It loads the libraries and defines the `check` helper used below.
import numpy as np                  # arrays + random numbers  (like base R vectors/matrices)
import matplotlib.pyplot as plt     # plotting                  (like ggplot2 / base plot)

def check(condition, hint):
    # Prints a green tick if `condition` is True, otherwise raises an error showing `hint`.
    if condition:
        print("✅ correct")
    else:
        raise AssertionError("❌ not yet — hint: " + hint)

print("numpy version:", np.__version__)

---
## 1. Python as a calculator

Recall the one-period binomial model from **Lecture 1 / Tutorial 1**: $S_0 = 4$, $u = 2$, $d = 1/2$, $r = 1/4$.
The risk-neutral probabilities are
$$\tilde p = \frac{1 + r - d}{u - d}, \qquad \tilde q = 1 - \tilde p = \frac{u-(1+r)}{u-d}.$$

In [ ]:
# DEMO — arithmetic
print(7 / 2)       # 3.5   ordinary division (always gives a decimal "float")
print(7 // 2)      # 3     integer division (R: 7 %/% 2)
print(2 ** 3)      # 8     POWER — use **
print(2 ^ 3)       # 1 (!) ^ is a bit operation in Python, NOT power. Classic bug for R users.
print(np.exp(1), np.sqrt(2), np.log(10))   # maths functions live in numpy

x = 0.1 + 0.2
print(x, x == 0.3, np.isclose(x, 0.3))     # floats are inexact -> compare with np.isclose

rate = 0.04
print(f"The rate is {rate}, or {rate:.1%}, and e^r = {np.exp(rate):.4f}")   # f-strings: like paste0/sprintf

**Exercise 1.1** — Store the model parameters and compute $\tilde p$ and $\tilde q$.

In [ ]:
S0 = 4
u  = ____
d  = ____
r  = ____

p_tilde = ____
q_tilde = ____

print(f"p_tilde = {p_tilde}, q_tilde = {q_tilde}")

In [ ]:
# CHECK 1.1
check(np.isclose(p_tilde, 0.5) and np.isclose(q_tilde, 0.5),
      "Tutorial 1 got p = q = 1/2. Did you use ** or ^? Brackets around (1 + r - d)?")

---
## 2. Functions

A function packages up a calculation so you can reuse it with different inputs (you'll price with $N = 10, 50, 100, \dots$ later — you do **not** want to copy-paste code for each $N$).

Python rules: `def name(inputs):`, then an **indented** body (4 spaces — indentation *is* the braces `{}`), and `return` sends the answer back. Unlike R, Python does **not** return the last line automatically — no `return`, no answer.

In [ ]:
# DEMO — a function with a default argument and a docstring
def discount(amount, r=0.04, T=1):
    '''Present value of `amount` paid at time T, with continuous compounding.'''
    return amount * np.exp(-r * T)

print(discount(100))              # uses the defaults r=0.04, T=1
print(discount(100, r=0.10))      # override one default by name
print(discount(100, 0.10, 2))     # or by position

# A function can return several things at once (a "tuple"), and you can unpack them:
def min_and_max(a, b):
    return min(a, b), max(a, b)

lo, hi = min_and_max(7, 3)
print(lo, hi)

**Exercise 2.1** — Turn your Section 1 formula into a function that returns **both** probabilities.

In [ ]:
def risk_neutral_probs(u, d, r):
    p = ____
    q = ____
    return ____

p, q = risk_neutral_probs(2, 0.5, 0.25)
print(p, q)

In [ ]:
# CHECK 2.1
check(np.allclose(risk_neutral_probs(2, 0.5, 0.25), (0.5, 0.5)) and
      np.allclose(risk_neutral_probs(1.2, 0.9, 0.0), (1/3, 2/3)),
      "p = (1 + r - d)/(u - d). The 2nd test is Lecture 2's upper branch (u=6/5, d=9/10, r=0 -> p=1/3). Did you `return p, q`?")

**Exercise 2.2** — Payoff functions. A call pays $(S - K)^+ = \max(S-K, 0)$; a put pays $(K - S)^+$.
Python's built-in `max(a, b)` works on two numbers.

In [ ]:
def call_payoff(S, K):
    return ____

def put_payoff(S, K):
    return ____

print(call_payoff(8, 5), call_payoff(2, 5), put_payoff(2, 5))

In [ ]:
# CHECK 2.2
check(call_payoff(8, 5) == 3 and call_payoff(2, 5) == 0 and put_payoff(2, 5) == 3 and put_payoff(8, 5) == 0,
      "Tutorial 1: V1(H) = (8-5)^+ = 3, V1(T) = (2-5)^+ = 0, P1(T) = (5-2)^+ = 3.")

**Exercise 2.3** — One-period risk-neutral pricing (Tutorial 1):
$$V_0 = \frac{1}{1+r}\left(\tilde p\, V_1(H) + \tilde q\, V_1(T)\right).$$

New idea: **functions are objects**, so you can pass `call_payoff` or `put_payoff` *into* another function as the argument `payoff`. Inside, `payoff(S, K)` calls whichever one was passed in. This is how one pricer will handle both calls and puts.

In [ ]:
def one_period_price(S0, u, d, r, K, payoff):
    S_up   = ____
    S_down = ____
    V_up   = ____
    V_down = ____
    p, q   = ____        # reuse your function from 2.1!
    return ____

C0 = one_period_price(4, 2, 0.5, 0.25, 5, call_payoff)
P0 = one_period_price(4, 2, 0.5, 0.25, 5, put_payoff)
print(f"call = {C0:.4f}, put = {P0:.4f}")

In [ ]:
# CHECK 2.3  — Tutorial 1 answer is $1.20 for both, and put-call parity C0 - P0 = S0 - K/(1+r) = 0
check(np.isclose(C0, 1.2) and np.isclose(P0, 1.2) and np.isclose(C0 - P0, 4 - 5/1.25),
      "V0 = (p*V_up + q*V_down)/(1+r). Pass the function itself (call_payoff), not call_payoff().")

**Exercise 2.4** — The replicating (delta-hedging) position: $\Delta_0 = \dfrac{V_1(H) - V_1(T)}{S_1(H) - S_1(T)}$.
Write it for any payoff, then check it gives $+\tfrac12$ for the call and $-\tfrac12$ for the put (Tutorial 1).

In [ ]:
def one_period_delta(S0, u, d, K, payoff):
    return ____

print(one_period_delta(4, 2, 0.5, 5, call_payoff), one_period_delta(4, 2, 0.5, 5, put_payoff))

In [ ]:
# CHECK 2.4
check(np.isclose(one_period_delta(4, 2, 0.5, 5, call_payoff), 0.5) and
      np.isclose(one_period_delta(4, 2, 0.5, 5, put_payoff), -0.5),
      "Numerator: payoff in the up state minus payoff in the down state. Denominator: S_up - S_down.")

---
## 3. NumPy arrays (Python's version of R vectors)

Plain Python **lists** `[1, 2, 3]` are *not* vectors: `+` glues lists together and `*` repeats them.
For maths you want **NumPy arrays**, which behave like R vectors (elementwise arithmetic, recycling a scalar, etc.).

**Indexing** starts at 0, and slices `a[start:stop]` *exclude* `stop`. So `a[1:3]` is elements 1 and 2 — the 2nd and 3rd.

In [ ]:
# DEMO — lists vs arrays
lst = [1, 2, 3]
print(lst + lst, lst * 2)            # [1, 2, 3, 1, 2, 3] twice — NOT maths!

a = np.array([10, 20, 30, 40, 50])
print(a + a, a * 2, a ** 2)          # elementwise, like R
print(a[0], a[-1], a[1:3], a[:2], a[2:])   # first, last, slice, first two, from index 2 on
print(len(a), a.shape, a.sum(), a.mean())

print(np.arange(5))                  # [0 1 2 3 4]        like 0:4 in R
print(np.arange(1, 6))               # [1 2 3 4 5]        like 1:5
print(np.linspace(0, 1, 5))          # [0. .25 .5 .75 1.]  like seq(0, 1, length.out = 5)

# max() vs np.maximum(): max(x) = the single largest; np.maximum(x, 0) = elementwise, like R's pmax
x = np.array([-2, 3, -1, 5])
print(max(x), np.maximum(x, 0))

**Exercise 3.1** — A time grid on $[0, 1]$ with $N = 10$ steps. It has $N+1$ points $0, 0.1, \dots, 1$ and spacing $\Delta t = 1/N$.
(This is exactly the grid you'll use for the random walks in Part I.)

In [ ]:
N  = 10
dt = ____
t  = ____
print(t, dt)

In [ ]:
# CHECK 3.1
check(len(t) == 11 and np.isclose(t[1], 0.1) and t[0] == 0 and t[-1] == 1 and np.isclose(dt, 0.1),
      "N steps need N + 1 grid points (the fence-post problem).")

**Exercise 3.2** — The **terminal nodes** of an $n$-period binomial tree. After $n$ steps with $j$ up-moves,
$$S_n(j) = S_0\, u^{j}\, d^{\,n-j}, \qquad j = 0, 1, \dots, n.$$
Build them all at once — no loop — by making `j` an array. Use Lecture 2's 3-period tree ($S_0 = 4$, $u = 2$, $d = 1/2$), which should give $\tfrac12, 2, 8, 32$.
Then compute the call payoffs with $K = 6$ using `np.maximum` (Lecture 2 got $0, 0, 2, 26$).

In [ ]:
n  = 3
j  = ____                 # array [0, 1, 2, 3] = number of up moves
S_T = ____          # S0, u, d are still 4, 2, 0.5 from Section 1
payoff_T = ____
print(S_T, payoff_T)

In [ ]:
# CHECK 3.2
check(np.allclose(S_T, [0.5, 2, 8, 32]) and np.allclose(payoff_T, [0, 0, 2, 26]),
      "j should be np.arange(n + 1); payoff uses np.maximum (not max) because S_T is an array.")

---
## 4. Random numbers

Modern NumPy style: create **one random generator** with a seed, then call methods on it. Same seed → same numbers every run (reproducible report!).

In [ ]:
# DEMO
rng = np.random.default_rng(2503)          # like set.seed(2503)

print(rng.normal(0, 1, size=5))            # 5 draws from N(0,1)       R: rnorm(5)
print(rng.normal(0, np.sqrt(0.1), size=3)) # N(0, 0.1): numpy wants the STANDARD DEVIATION, like R
print(rng.choice([-1, 1], size=10))        # fair ±1 coin flips        R: sample(c(-1,1), 10, replace=TRUE)
print(rng.uniform(size=3))                 # U(0,1)                    R: runif(3)

**Exercise 4.1** — A symmetric random walk (Lecture 3): $Z_i \overset{iid}{\sim} \pm 1$ with prob. $\tfrac12$, and $X_n = \sum_{i=1}^n Z_i$.
Draw 10 steps and compute the whole path $X_1, \dots, X_{10}$ with one function call.

In [ ]:
rng = np.random.default_rng(1)
Z = ____
X = ____
print("steps:", Z)
print("path: ", X)

In [ ]:
# CHECK 4.1
check(len(X) == 10 and set(np.unique(Z)) <= {-1, 1} and X[-1] == Z.sum() and np.all(np.abs(np.diff(X)) == 1),
      "Z: rng.choice with the two values -1 and 1. X: the running (cumulative) sum of Z.")

---
## 5. Loops

`for i in range(n):` runs with `i = 0, 1, …, n-1`. To count **backwards** (backward induction!) use `range(start, stop, step)` with step `-1` — the stop is still excluded, so `range(3, -1, -1)` gives 3, 2, 1, 0.

In NumPy, loops are usually slower than vectorised code — but for the binomial tree, a loop over *time steps* is natural and fine.

In [ ]:
# DEMO
for i in range(4):
    print("forward", i)
for k in range(3, -1, -1):
    print("backward", k)

total = 0
for z in [5, -1, 2]:        # you can also loop directly over the items
    total = total + z       # (or: total += z)
print(total)

**Exercise 5.1** — Rebuild the random-walk path from 4.1 **with a loop**, starting at $X_0 = 0$.
`X_loop` has 11 entries: `X_loop[0] = 0`, and each next entry adds one step.

In [ ]:
X_loop = np.zeros(11)                  # [0, 0, ..., 0]  — 11 slots
for i in range(10):
    X_loop[i + 1] = ____
print(X_loop)

In [ ]:
# CHECK 5.1 — X_loop should equal [0, then the cumsum from 4.1]
check(np.allclose(X_loop, np.concatenate([[0], X])),
      "Entry i+1 = entry i plus step Z[i].")

**Exercise 5.2 — Challenge (preview of Part II).** Price Lecture 2's 3-period European call ($S_0=4$, $u=2$, $d=\tfrac12$, $r=\tfrac14$, $K=6$) by **backward induction**. Your notes got $V_0 = 2.048$.

Start from `V = payoff_T` (ordered $j = 0,1,2,3$ up-moves). At each step back, the node with $j$ up-moves has its **up** child at position $j+1$ and its **down** child at position $j$. With slicing that is:
- up children:   `V[1:]`  (drop the first)
- down children: `V[:-1]` (drop the last)

so one step back is `V = (p * V[1:] + q * V[:-1]) / (1 + r)` — each step shortens `V` by one, until a single number is left.

In [ ]:
p, q = risk_neutral_probs(2, 0.5, 0.25)
V = payoff_T.copy()          # .copy() so we don't overwrite payoff_T
for step in ____:    # 3 steps back: 3 -> 2 -> 1 -> 0
    V = ____
    print(f"after stepping back from time {step}: {V}")
V0 = V[0]

In [ ]:
# CHECK 5.2
check(np.isclose(V0, 2.048),
      "You need exactly n = 3 backward steps. Compare your printed time-2 values with Lecture 2: V2 = 0, 0.8, 11.2.")

---
## 6. 2-D arrays: many paths at once

Simulations store **one path per row, one time point per column**, i.e. shape `(n_paths, n_steps)`.
The `axis` argument says which direction to work along:
- `axis=1` → along each **row** (across time) — e.g. `np.cumsum(Z, axis=1)` builds each path.
- `axis=0` → down each **column** (across paths) — e.g. `X.mean(axis=0)` is the average path.

In [ ]:
# DEMO
M = np.arange(12).reshape(3, 4)     # 3 rows, 4 columns   R: matrix(0:11, nrow=3, byrow=TRUE)
print(M)
print(M.shape)                      # (3, 4)
print(M[0])                         # first row
print(M[:, 0])                      # first column  (":" means "all rows")
print(M[:, -1])                     # last column
print(np.cumsum(M, axis=1))         # running sum along each row
print(M.sum(axis=0), M.sum(axis=1))

# Glue a column of zeros on the left — this is how we add the starting value X_0 = 0 to every path
print(np.hstack([np.zeros((3, 1)), M]))

**Exercise 6.1** — Simulate `n_paths = 5` random walks, each with `n_steps = 10`, stored as a `(5, 11)` array that starts with a column of zeros.

In [ ]:
rng = np.random.default_rng(42)
n_paths, n_steps = 5, 10

Z = ____           # shape (5, 10)
X = ____                                   # each row = one path
X = ____                 # add X_0 = 0 -> shape (5, 11)
print(X.shape)
print(X)

In [ ]:
# CHECK 6.1
check(X.shape == (5, 11) and np.all(X[:, 0] == 0) and np.all(np.abs(np.diff(X, axis=1)) == 1),
      "size=(n_paths, n_steps) in rng.choice; cumsum along axis=1; then hstack a (n_paths, 1) column of zeros.")

**Exercise 6.2 — Monte Carlo check of Lecture 3.** For a symmetric random walk, $\mathbb E[X_n] = 0$ and $\operatorname{Var}[X_n] = n$.
Simulate 100,000 walks of $n = 20$ steps and estimate both from the **last column**. (You don't need the zero column here.)

In [ ]:
rng = np.random.default_rng(0)
Z_big = ____      # underscores in numbers are allowed: 100_000
X_big = ____
X_n   = ____                                 # the value at time n for every path
print(f"mean = {X_n.mean():.3f}  (theory 0),   var = {X_n.var():.3f}  (theory 20)")

In [ ]:
# CHECK 6.2
check(X_n.shape == (100_000,) and abs(X_n.mean()) < 0.1 and abs(X_n.var() - 20) < 0.5,
      "X_n should be the LAST column of the cumulative sums: X_big[:, -1].")

---
## 7. Plotting with matplotlib

Pattern you'll reuse for every figure in the report:
```python
fig, ax = plt.subplots(figsize=(7, 4))   # a figure with one set of axes
ax.plot(x, y)                            # draw
ax.set_xlabel(...); ax.set_ylabel(...); ax.set_title(...)
fig.savefig("figures/name.pdf", bbox_inches="tight")   # PDF drops straight into your LaTeX template
plt.show()
```
`ax.plot(t, Y)` with a 2-D `Y` draws **one line per column** — so to draw one line per *path* (rows), pass the transpose `Y.T`.

In [ ]:
# DEMO
tt = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(tt, np.sin(tt), label="sin")
ax.plot(tt, np.cos(tt), label="cos", linestyle="--")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("t"); ax.set_ylabel("value"); ax.set_title("Demo plot")
ax.legend()
plt.show()

**Exercise 7.1 — Bridge to Part I.** The **scaled** random walk from Lecture 3 is
$$W^{(N)}_t = \frac{1}{\sqrt N}\sum_{i=1}^{Nt} Z_i ,\qquad t \in \{0, \tfrac1N, \tfrac2N, \dots, 1\}.$$
With $N = 10$ it is just your 6.1 paths divided by $\sqrt{10}$, on the time grid from 3.1. Plot the 5 paths.

In [ ]:
N = 10
t = ____
W = ____               # X is your (5, 11) array from 6.1

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(____, ____)
ax.set_xlabel("t")
ax.set_ylabel(r"$W^{(N)}_t$")      # r"..." lets you write LaTeX in labels
ax.set_title(f"Scaled symmetric random walk, N = {N}")
plt.show()

In [ ]:
# CHECK 7.1
check(W.shape == (5, 11) and np.allclose(W * np.sqrt(10), X) and len(t) == 11,
      "W = X / sqrt(N), and t has N + 1 points. Plot W.T so each PATH is a line.")

---
## 🎉 Done — what you now know
`**` for powers · functions with `return` · passing functions as arguments · arrays, 0-indexing, slicing · `np.linspace` / `np.arange` / `np.cumsum` / `np.maximum` · `rng.choice` / `rng.normal` · forward & backward loops · 2-D arrays and `axis` · plotting paths.

**That is almost everything Project 1 needs.** Next is **Notebook 01 — Part I**, where you will:
1. write a function `scaled_random_walk(N, n_paths, rng)` from your 6.1/7.1 code,
2. simulate Brownian motion with Gaussian increments $W_{t+\Delta t} - W_t \sim \mathcal N(0, \Delta t)$ (hint: standard deviation $\sqrt{\Delta t}$!) with $N_1 = 10{,}000$,
3. compare $N_2 = 10, 50, 200, 1000$ side by side and save the figure for the report.

When you've finished, tell me which exercises felt shaky and I'll adjust Notebook 01.